# 02 · Baselines

How good is a forecast that needs no model? These are the numbers LightGBM must beat.

In [ ]:
import pathlib
import sys

ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
import os

os.chdir(ROOT)
import pandas as pd

pd.set_option("display.width", 160)

In [ ]:
from velib import train

data, stations, centroids = train.load_dataset()
train_df, test_df, cutoff = train.time_split(data)
print("cutoff:", cutoff, "| train rows:", len(train_df), "| test rows:", len(test_df))

In [ ]:
from sklearn.metrics import mean_absolute_error

for h in ("1h", "2h"):
    t = test_df.dropna(subset=[f"bikes_{h}"])
    print(
        h,
        "persistence MAE:",
        round(mean_absolute_error(t[f"bikes_{h}"], t["bikes_now"]), 3),
        "| seasonal MAE:",
        round(mean_absolute_error(t[f"bikes_{h}"], train.seasonal_baseline(t, h)), 3),
    )

Persistence error by hour of day: forecasting is hardest during the rush hours.

In [ ]:
t = test_df.dropna(subset=["bikes_2h"]).assign(err=lambda d: (d["bikes_2h"] - d["bikes_now"]).abs())
t.groupby(t["hour"].astype(int))["err"].mean().plot(
    kind="bar", title="Persistence MAE (+2h) by hour", figsize=(9, 3.5)
);